# CXR-Intel: Multi-Modal Chest X-Ray Intelligence System
## Full Pipeline on Kaggle GPU
**Models:** MedGemma 1.5-4b-it + CLIP + ColPali  
**Mode:** Real inference (no mock) — GPU T4 16GB VRAM

### Instructions:
1. Make sure GPU is enabled: Settings → Accelerator → GPU T4 x2
2. Add your HF token: Add-ons → Secrets → HF_TOKEN
3. Add MIMIC CXR dataset via **Add Data** (already done ✅)
4. **No need to upload project zip** — code is cloned from GitHub automatically
5. Run all cells top to bottom

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q transformers==4.47.1
!pip install -q colpali-engine
!pip install -q faiss-cpu
!pip install -q rouge-score
!pip install -q nltk
!pip install -q bert-score
!pip install -q open_clip_torch
!pip install -q sentence-transformers
!pip install -q python-dotenv
!pip install -q accelerate
!pip install -q bitsandbytes
!pip install -q pyyaml
print('All packages installed!')

## Cell 2 — Setup Environment + Clone Project from GitHub

In [ ]:
import os, shutil, subprocess

# ── Load HF token from Kaggle Secrets ──
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HF_TOKEN loaded from Kaggle secrets')
except Exception as e:
    print(f'Warning: HF_TOKEN not found in secrets: {e}')
    HF_TOKEN = ''
    os.environ['HF_TOKEN'] = HF_TOKEN

os.environ['USE_MOCK_MODE'] = 'false'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['SAMPLE_SIZE'] = '300'

# ── Check if already inside the repo (GitHub import puts us here) ──
WORK_DIR = '/kaggle/working/cxr-intel'
GITHUB_REPO = 'https://github.com/HAMZA2FEKRY/cxr-intel.git'

# Check if key repo files are in /kaggle/working already (GitHub import)
already_in_repo = os.path.exists('/kaggle/working/scripts') and \
                  os.path.exists('/kaggle/working/rag') and \
                  os.path.exists('/kaggle/working/models')

if already_in_repo:
    WORK_DIR = '/kaggle/working'
    os.chdir(WORK_DIR)
    print(f'Already in repo (GitHub import mode): {WORK_DIR}')
elif os.path.exists(WORK_DIR):
    print('Repo already cloned, pulling latest...')
    result = subprocess.run(['git', '-C', WORK_DIR, 'pull'],
                            capture_output=True, text=True)
    print(result.stdout or 'Already up to date')
    os.chdir(WORK_DIR)
else:
    print(f'Cloning from GitHub: {GITHUB_REPO}')
    result = subprocess.run(['git', 'clone', GITHUB_REPO, WORK_DIR],
                            capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f'Clone failed: {result.stderr}')
    os.chdir(WORK_DIR)

print(f'Working dir: {os.getcwd()}')
print('Contents:', sorted(os.listdir('.')))

## Cell 3 — Create config.yaml

In [ ]:
import yaml

config = {
    'dataset': {
        'kaggle_slug': 'simhadrisadaram/mimic-cxr-dataset',
        'raw_dir': 'data/raw/mimic_cxr',
        'raw_train_csv': 'data/raw/mimic_cxr/mimic_cxr_aug_train.csv',
        'processed_dir': 'data/processed',
        'sample_size': 300,
        'sample_metadata_path': 'data/samples/sample_metadata.csv'
    },
    'columns': {
        'report_candidates': ['text', 'report', 'findings', 'impression'],
        'image_candidates': ['image_path', 'path', 'filename', 'image', 'dicom_id'],
        'id_candidates': ['image_id', 'subject_id', 'study_id', 'dicom_id', 'id']
    },
    'rag': {
        'top_k': 3,
        'index_dir': 'rag/indexes'
    },
    'models': {
        'medgemma_model_id': 'google/medgemma-1.5-4b-it',
        'use_mock_mode': False
    },
    'app': {
        'title': 'CXR-Intel'
    }
}

os.makedirs('configs', exist_ok=True)
with open('configs/config.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print('config.yaml written:')
print(open('configs/config.yaml').read())

## Cell 4 — Link MIMIC CXR Dataset + Prepare Data

In [ ]:
import shutil, glob

# ── Scan all of /kaggle/input/ for the MIMIC dataset ──
print('Scanning /kaggle/input/ for datasets...')
all_inputs = []
try:
    all_inputs = os.listdir('/kaggle/input')
    print('Found input datasets:', all_inputs)
except Exception as e:
    print(f'Error listing /kaggle/input: {e}')

# ── Find MIMIC input directory (flexible — matches any variant) ──
MIMIC_INPUT = None
for d in all_inputs:
    if 'mimic' in d.lower() or 'cxr' in d.lower():
        MIMIC_INPUT = f'/kaggle/input/{d}'
        print(f'Using MIMIC dataset at: {MIMIC_INPUT}')
        break

if MIMIC_INPUT is None:
    print('\nERROR: No MIMIC/CXR dataset found in /kaggle/input/')
    print('Steps to fix:')
    print('  1. Click "Add Input" in the right panel')
    print('  2. Search for: simhadrisadaram/mimic-cxr-dataset')
    print('  3. Click Add, then re-run this cell')
else:
    # ── Show first 15 files ──
    print('\nFirst 15 files in dataset:')
    shown = 0
    for root, dirs, files in os.walk(MIMIC_INPUT):
        for f in files:
            print(f'  {os.path.join(root, f)}')
            shown += 1
            if shown >= 15: break
        if shown >= 15: break

    # ── Find and copy the train CSV ──
    os.makedirs('data/raw/mimic_cxr', exist_ok=True)
    csv_found = False
    for root, dirs, files in os.walk(MIMIC_INPUT):
        for f in files:
            if f.endswith('.csv'):
                src = os.path.join(root, f)
                dst = 'data/raw/mimic_cxr/mimic_cxr_aug_train.csv'
                shutil.copy2(src, dst)
                print(f'\nCSV copied: {f} -> {dst}')
                csv_found = True
                break
        if csv_found: break

    if not csv_found:
        print('WARNING: No CSV found in dataset')

    # ── Copy images (up to 300) ──
    os.makedirs('data/samples/images', exist_ok=True)
    image_count = 0
    for root, dirs, files in os.walk(MIMIC_INPUT):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                src = os.path.join(root, f)
                dst = os.path.join('data/samples/images', f)
                if not os.path.exists(dst):
                    shutil.copy2(src, dst)
                    image_count += 1
            if image_count >= 300: break
        if image_count >= 300: break

    print(f'Images copied: {image_count}')
    print('\nData setup complete!')

## Cell 5 — Prepare Dataset + Generate QA Pairs

In [ ]:
import sys
sys.path.insert(0, WORK_DIR)

print('='*50)
print('INSPECTING DATASET')
print('='*50)
!python scripts/inspect_dataset.py

print('='*50)
print('PREPARING DATASET')
print('='*50)
!python scripts/prepare_dataset.py

print('='*50)
print('CREATING SAMPLE')
print('='*50)
!python scripts/create_sample.py

print('='*50)
print('GENERATING QA DATASET')
print('='*50)
!python scripts/generate_qa_dataset.py

print('Data preparation complete!')

## Cell 6 — Load Models + Build Indexes

In [ ]:
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print('='*50)
print('BUILDING CLIP INDEX')
print('='*50)
!python rag/build_index.py --model clip

print('='*50)
print('BUILDING COLPALI INDEX')
print('='*50)
!python rag/build_index.py --model colpali

print('Indexes built!')

## Cell 7 — Run Report Generation (Real MedGemma)

In [ ]:
print('='*50)
print('RUNNING REPORT GENERATION')
print('This will load MedGemma 1.5-4b-it (~8GB VRAM)')
print('='*50)
!python scripts/run_report_generation.py

import pandas as pd
df = pd.read_csv('results/report_generation_results.csv')
print(f'\nGenerated {len(df)} reports')
print(f'Mode used: {df["mode_used"].iloc[0]}')
print('\nSample report:')
print(df['generated_report'].iloc[0][:300])

## Cell 8 — Run QA Pipeline

In [ ]:
print('='*50)
print('RUNNING QA PIPELINE')
print('CLIP + ColPali retrieve -> MedGemma answers')
print('='*50)
!python scripts/run_qa_pipeline.py

df_qa = pd.read_csv('results/qa_results.csv')
print(f'\nAnswered {len(df_qa)} questions')
print('\nSample QA:')
row = df_qa.iloc[0]
print(f'Q: {row["question"]}')
print(f'A (CLIP): {row["clip_answer"][:150]}')

## Cell 9 — Evaluate Everything + Compare Models

In [ ]:
print('='*50)
print('EVALUATING REPORTS')
print('='*50)
!python evaluation/evaluate_reports.py

print('='*50)
print('EVALUATING RETRIEVAL')
print('='*50)
!python evaluation/evaluate_retrieval.py

print('='*50)
print('EVALUATING QA')
print('='*50)
!python evaluation/evaluate_qa.py

print('='*50)
print('MODEL COMPARISON')
print('='*50)
!python evaluation/compare_models.py

import json
print('\n' + '='*50)
print('FINAL RESULTS SUMMARY')
print('='*50)
for fname in ['report_generation_metrics.json', 'retrieval_metrics.json', 'qa_metrics.json']:
    fpath = f'results/{fname}'
    if os.path.exists(fpath):
        print(f'\n{fname}:')
        print(json.dumps(json.load(open(fpath)), indent=2))

## Cell 10 — Save + Download Results

In [ ]:
import shutil

shutil.make_archive('/kaggle/working/cxr_intel_results', 'zip', WORK_DIR, 'results')
print('Results zipped: /kaggle/working/cxr_intel_results.zip')

shutil.make_archive('/kaggle/working/cxr_intel_indexes', 'zip', WORK_DIR, 'rag/indexes')
print('Indexes zipped: /kaggle/working/cxr_intel_indexes.zip')

shutil.make_archive('/kaggle/working/cxr_intel_images', 'zip', WORK_DIR, 'data/samples')
print('Images zipped: /kaggle/working/cxr_intel_images.zip')

print('\nDownload these from the Kaggle Output tab:')
print('  cxr_intel_results.zip  -> extract to your local results/ folder')
print('  cxr_intel_indexes.zip  -> extract to your local rag/indexes/ folder')
print('  cxr_intel_images.zip   -> extract to your local data/samples/ folder')
print('\nThen run locally: streamlit run app/streamlit_app.py')

## Cell 11 — Visual Summary of Results

In [ ]:
import matplotlib.pyplot as plt
import json

retrieval = json.load(open('results/retrieval_metrics.json'))
qa = json.load(open('results/qa_metrics.json'))
report = json.load(open('results/report_generation_metrics.json'))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('CXR-Intel Model Comparison Results', fontsize=14, fontweight='bold')

ax1 = axes[0]
ks = ['Retrieval@1', 'Retrieval@3', 'Retrieval@5']
clip_vals = [retrieval['clip'][k] for k in ks]
colpali_vals = [retrieval['colpali'][k] for k in ks]
x = range(len(ks))
ax1.bar([i-0.2 for i in x], clip_vals, 0.4, label='CLIP', color='steelblue')
ax1.bar([i+0.2 for i in x], colpali_vals, 0.4, label='ColPali', color='coral')
ax1.set_title('Retrieval Performance')
ax1.set_xticks(list(x))
ax1.set_xticklabels(['@1', '@3', '@5'])
ax1.set_ylabel('Score')
ax1.legend()
ax1.set_ylim(0, 1)

ax2 = axes[1]
metrics = ['ROUGE-L', 'BLEU']
vals = [report['rougeL'], report['bleu']]
ax2.bar(metrics, vals, color=['mediumseagreen', 'mediumpurple'])
ax2.set_title('Report Generation (MedGemma)')
ax2.set_ylabel('Score')
ax2.set_ylim(0, 1)
for i, v in enumerate(vals):
    ax2.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=11)

ax3 = axes[2]
qa_labels = ['CLIP+MedGemma', 'ColPali+MedGemma']
qa_vals = [qa['clip_similarity'], qa['colpali_similarity']]
ax3.bar(qa_labels, qa_vals, color=['steelblue', 'coral'])
ax3.set_title('QA Token Similarity')
ax3.set_ylabel('Score')
ax3.set_ylim(0, 1)
for i, v in enumerate(qa_vals):
    ax3.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=11)

plt.tight_layout()
plt.savefig('results/model_comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to results/model_comparison_chart.png')